# Results

## Get data

In [6]:
import time

import numpy as np
import pandas as pd
import plotly.express as px
import wandb
from transformers import AutoTokenizer

from helpers import get_data_from_file

template = "plotly_white"

fancy_cols = {'corr2incorr':            {"name": "Correct ⟶ Incorrect", "asc": True},
              'peak_ram_memory_mb':     {"name": "Peak RAM memory (MB)", "asc": True},
              'accuracy_sentences':     {"name": "Accuracy (sentences)", "asc": False},
              'gpu_memory_mb':          {"name": 'GPU memory (MB)', "asc": True},
              'incorr2incorr':          {"name": "Incorrect ⟶ Incorrect", "asc": True},
              'word_incorrection_rate': {"name": "Word incorrection rate", "asc": True},
              'ms_per_sentence':        {"name": "Inference time (ms/sentence)", "asc": True},
              'throughput_words':       {"name": "Throughput (words/s)", "asc": False},
              'accuracy_words':         {"name": "Accuracy (words)", "asc": False},
              'recall':                 {"name": "Recall", "asc": False},
              'incorr2corr':            {"name": "Incorrect ⟶ Correct", "asc": False},
              'corr2corr':              {"name": "Correct ⟶ Correct", "asc": False},
              'f05':                    {"name": "F0.5", "asc": False},
              'precision':              {"name": "Precision", "asc": False},
              'model_size':             {"name": "Model size (MB)", "asc": True},
              }

api = wandb.Api()
runs = api.runs("martin-elias-ctu-fit/Benchmarks")

run_dfs = []
for run in runs:
    run_df = run.history(keys=None)
    run_df["name"] = run.name
    # Rename old run metrics from token to word
    run_df.rename(columns={
            "throughput_tokens":       "throughput_words",
            "token_incorrection_rate": "word_incorrection_rate",
            "token_correction_rate":   "word_correction_rate",
            "accuracy_tokens":         "accuracy_words",

    }, inplace=True)
    run_dfs.append(run_df)
df = pd.concat(run_dfs, axis=0)

# Drop anything I don't care about in the graph
df.drop(columns=["_runtime", "_step", "_timestamp", "model_name", "skipped", 'ram_memory_mb', "should_skip",
                 "word_correction_rate"], inplace=True)
df.loc[df.name == "jamspell", "model_size"] = 35

## BERT

In [12]:
bert_df = df[df["name"].str.contains("bert", case=False, na=False)].copy()
bert_df["name"] = bert_df["name"].str.replace("bert-checker-", "", case=False, regex=False).str.strip()
bert_df.loc[bert_df.name == "finetuned", "name"] = "Fine-tuned"
bert_df.loc[bert_df.name == "pretrained", "name"] = "Pre-trained"
bert_df.loc[bert_df.name == "pretrained-wo space correction", "name"] = "Pre-trained - no space correction"
bert_df.loc[bert_df.name == "finetuned-wo space correction", "name"] = "Fine-tuned - no space correction"

In [ ]:
for col in bert_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_bert_df = bert_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_bert_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='')
        fig.show()


In [84]:
mx = "ms_per_sentence"
my = "accuracy_words"

fig = px.scatter(
        bert_df,
        x=mx,
        y=my,
        color="name",
        # text='name',
        title="NeuSpell BERT - Accuracy vs Speed<br>(point size = word incorrection rate, smaller is better)",
        size="word_incorrection_rate",
        # labels={"ms_per_sentence": "Inference Time (ms)", "accuracy_words": "Model Accuracy", "name": "Version"},
        template=template
)
fig.update_traces(textposition=['bottom center', 'bottom center', 'top left', 'top right'], )
fig.update_layout(
        title_x=0.5,
        yaxis=dict(range=[0, None]),
        xaxis_title="Inference Time (ms, lower is better)",
        yaxis_title="Words accuracy (higher is better)",
        xaxis=dict(autorange="reversed"),
        showlegend=False,
        font=dict(size=15),
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Pre-trained", "ms_per_sentence"][0],
        y=bert_df.loc[bert_df.name == "Pre-trained", "accuracy_words"][0],
        text="Pre-trained",
        showarrow=False,
        font=dict(size=15),
        xshift=0, yshift=-20
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Fine-tuned", "ms_per_sentence"][0],
        y=bert_df.loc[bert_df.name == "Fine-tuned", "accuracy_words"][0],
        text="Fine-tuned",
        showarrow=False,
        font=dict(size=15),
        xshift=0, yshift=-20
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Pre-trained - no space correction", "ms_per_sentence"][0],
        y=bert_df.loc[bert_df.name == "Pre-trained - no space correction", "accuracy_words"][0],
        text="Pre-trained<br>no space correction",
        showarrow=False,
        font=dict(size=15),
        xshift=-40, yshift=-30
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Fine-tuned - no space correction", "ms_per_sentence"][0],
        y=bert_df.loc[bert_df.name == "Fine-tuned - no space correction", "accuracy_words"][0],
        text="Fine-tuned<br>no space correction",
        showarrow=False,
        font=dict(size=15),
        xshift=40, yshift=30
)
fig.show()

In [85]:
fig.write_image(file="../thesis/images/bert_accVSspeed.pdf", width=960, height=540, engine="kaleido")

## ELMO

In [7]:
elmo_df = df[df["name"].str.contains("elmo", case=False, na=False)].copy()
elmo_df["name"] = elmo_df["name"].str.replace("elmo-checker-", "", case=False, regex=False).str.strip()
elmo_df.loc[elmo_df.name == "finetuned", "name"] = "Fine-tuned"
elmo_df.loc[elmo_df.name == "pretrained", "name"] = "Pre-trained"
elmo_df.loc[elmo_df.name == "pretrained-wo space correction", "name"] = "Pre-trained - no space correction"
elmo_df.loc[elmo_df.name == "finetuned-wo space correction", "name"] = "Fine-tuned - no space correction"

In [14]:
for col in elmo_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_elmo_df = elmo_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_elmo_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='')
        fig.show()


## ALL graphs

In [ ]:
# Get all the graphs
for col in df:
    if col in ["name", "inference_time", "throughput_sentences", 'typo_detection_model_inference_time',
               'typo_detection_model_ms_per_sentence', ]:
        continue
    sorted_df = df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
    fig = px.bar(
            sorted_df,
            x=col,
            y="name",
            color="name",
            title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
            labels={"name": "Model"},
    )
    fig.update_layout(title_x=0.5, xaxis_title='')
    fig.show()

## Detect typo tokenizer max len

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nreimers/MiniLM-L6-H384-uncased")

allDF = []
for name in ['train', 'dev', 'test']:
    df, _ = get_data_from_file(name)
    allDF.extend(df)

print(len(allDF))

token_lengths = []
start = time.time()
for text in allDF:
    tokens = tokenizer.encode(text)
    token_lengths.append(len(tokens))
print(f"Time taken to tokenize: {time.time() - start:.2f} seconds")

df_tokens = pd.DataFrame({'Token Length': token_lengths})

In [ ]:
max_len = 96
fig = px.histogram(
        df_tokens,
        x="Token Length",
        nbins=50,
        title="Distribution of Token Lengths in Dataset",
        labels={"Token Length": "Number of Tokens"},
        template=template,
)
fig.add_vline(
        x=max_len,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Current max_len: {max_len}",
        annotation_position="top right"
)
fig.update_layout(
        xaxis_title="Number of Tokens",
        yaxis_title="Number of Sentences",
        bargap=0.1
)
fig.show()

# Statistics
print(f"Maximum token length: {max(token_lengths)}")
print(f"Mean token length: {np.mean(token_lengths):.2f}")
print(f"Median token length: {np.median(token_lengths)}")
print(f"95th percentile: {np.percentile(token_lengths, 95)}")
print(f"99th percentile: {np.percentile(token_lengths, 99)}")
print(f"Percentage of sentences truncated: {sum(l > max_len for l in token_lengths) / len(token_lengths):.2%}")
